# RFD Urban Digital Twin Analysis

This notebook builds compact exam pipeline for RFD-based consistency profiling on urban air-quality data.

Scope:
- simplified conceptual Urban Digital Twin;
- Beijing air-quality subset with two stations;
- interpretable approximate rules, not causal claims;
- full DiMε discovery, as specified in the KDIID course.


## Dataset Configuration

Selected subset:
- stations: `Aotizhongxin`, `Changping`
- period: `2013-03-01` to `2017-02-28`
- preprocessing and profiling use the complete common station coverage
- DiMε relation: weekly medians by station and time slot, built from every cleaned row

DiMε traverses the complete lattice of the eight projected discovery attributes. The legacy 1500-row sample is retained only for supplementary manual-candidate comparisons.


In [1]:
from pathlib import Path
import sys

import pandas as pd

try:
    from IPython.display import display
except ImportError:
    def display(obj):
        print(obj)

current = Path.cwd().resolve()
PROJECT_ROOT = next(path for path in [current, *current.parents] if (path / 'AGENTS.md').exists())
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

DATA_RAW = PROJECT_ROOT / 'data' / 'raw'
DATA_PROCESSED = PROJECT_ROOT / 'data' / 'processed'
RESULTS_DIR = PROJECT_ROOT / 'results'
FIGURES_DIR = PROJECT_ROOT / 'figures'

for path in [DATA_PROCESSED, RESULTS_DIR, FIGURES_DIR]:
    path.mkdir(parents=True, exist_ok=True)


In [2]:
from src.experiments import (
    prepare_dime_projection,
    prepare_rfd_sample,
    rules_from_discovery_frame,
    run_dime_discovery,
    run_candidate_validation,
    run_station_comparison,
    run_threshold_comparison,
    select_top_rule_violations,
    export_violation_examples,
    run_candidate_metrics,
    run_bootstrap_validation,
    run_train_test_validation,
    run_window_evolution,
    run_violation_analysis,
)
from src.preprocessing import prepare_udt_dataset
from src.profiling import (
    correlation_matrix,
    data_types_summary,
    dataset_shape_summary,
    hour_distribution,
    missing_values_summary,
    numeric_summary,
    station_distribution,
    time_slot_distribution,
    unique_values_summary,
)
from src.visualization import (
    plot_correlation_matrix,
    plot_missing_values,
    plot_pm25_timeseries,
    plot_pollutant_distribution,
)


## Step 1-2: Preprocessing

In [3]:
cleaned_df, pre_drop_missing = prepare_udt_dataset(
    raw_dir=DATA_RAW,
    processed_path=DATA_PROCESSED / 'udt_rfd_dataset.csv',
)
sample_df = prepare_rfd_sample(cleaned_df)
sample_df.to_csv(DATA_PROCESSED / 'udt_rfd_sample.csv', index=False)
dime_df = prepare_dime_projection(cleaned_df)
dime_df.to_csv(DATA_PROCESSED / 'udt_dime_projection.csv', index=False)

print('Cleaned rows:', len(cleaned_df))
print('Cleaned columns:', cleaned_df.shape[1])
print('DiMε projection rows:', len(dime_df))
print('Source rows represented:', int(dime_df['source_rows'].sum()))
print('Legacy supplementary sample rows:', len(sample_df))
display(pre_drop_missing)


Cleaned rows: 66619
Cleaned columns: 11
DiMε projection rows: 1676
Source rows represented: 66619
Legacy supplementary sample rows: 1500


,column,missing_count,missing_rate
0,O3,2323,0.033125
1,PM2.5,1699,0.024227
2,NO2,1690,0.024099
3,PM10,1300,0.018538
4,DEWP,73,0.001041
5,TEMP,73,0.001041
6,WSPM,57,0.000813
7,datetime,0,0.000000
8,hour,0,0.000000
9,station,0,0.000000


## Step 3: Profiling

In [4]:
shape_df = dataset_shape_summary(cleaned_df)
dtypes_df = data_types_summary(cleaned_df)
missing_df = missing_values_summary(cleaned_df)
unique_df = unique_values_summary(cleaned_df)
numeric_df = numeric_summary(cleaned_df)
station_df = station_distribution(cleaned_df)
hour_df = hour_distribution(cleaned_df)
time_slot_df = time_slot_distribution(cleaned_df)
corr_df = correlation_matrix(
    cleaned_df,
    numeric_columns=['hour', 'PM2.5', 'PM10', 'NO2', 'O3', 'TEMP', 'DEWP', 'WSPM'],
)

shape_df.to_csv(RESULTS_DIR / 'profile_shape_summary.csv', index=False)
dtypes_df.to_csv(RESULTS_DIR / 'profile_dtypes_summary.csv', index=False)
missing_df.to_csv(RESULTS_DIR / 'profile_missing_summary.csv', index=False)
unique_df.to_csv(RESULTS_DIR / 'profile_unique_values_summary.csv', index=False)
numeric_df.to_csv(RESULTS_DIR / 'profile_numeric_summary.csv', index=False)
station_df.to_csv(RESULTS_DIR / 'profile_station_distribution.csv', index=False)
hour_df.to_csv(RESULTS_DIR / 'profile_hour_distribution.csv', index=False)
time_slot_df.to_csv(RESULTS_DIR / 'profile_time_slot_distribution.csv', index=False)
corr_df.to_csv(RESULTS_DIR / 'profile_correlation_matrix.csv')

plot_missing_values(missing_df, FIGURES_DIR / 'missing_values.png')
plot_pollutant_distribution(cleaned_df, 'PM2.5', FIGURES_DIR / 'pm25_distribution.png')
plot_pollutant_distribution(cleaned_df, 'NO2', FIGURES_DIR / 'no2_distribution.png')
plot_pm25_timeseries(cleaned_df, FIGURES_DIR / 'pm25_timeseries.png')
plot_correlation_matrix(corr_df, FIGURES_DIR / 'correlation_matrix.png')

print('Dataset shape')
display(shape_df)
print('Data types')
display(dtypes_df)
print('Missing values after cleaning')
display(missing_df)
print('Unique values')
display(unique_df)
print('Numeric summary')
display(numeric_df)
print('Station distribution')
display(station_df)
print('Hour distribution')
display(hour_df)
print('Time-slot distribution')
display(time_slot_df)


Dataset shape


,metric,value
0,rows,66619
1,columns,11


Data types


,column,dtype
0,DEWP,float64
1,NO2,float64
2,O3,float64
3,PM10,float64
4,PM2.5,float64
5,TEMP,float64
6,WSPM,float64
7,datetime,datetime64[us]
8,hour,int64
9,station,str


Missing values after cleaning


...

,station,rows
0,Aotizhongxin,32779
1,Changping,33840


Hour distribution


,hour,rows
0,0,2813
1,1,2811
2,2,2740
3,3,2722
4,4,2435
5,5,2749
6,6,2809
7,7,2801
8,8,2810
9,9,2828


Time-slot distribution


,time_slot,rows
0,night,16270
1,morning,16823
2,afternoon,16645
3,evening,16881


## Step 4: RFD discovery algorithm: DiMε

The course algorithm uses per-attribute difference matrices, stripped similar subsets, a complete TANE-style attribute lattice, C+ candidate sets, DiMε pruning, and g3 extent validation. The default g3 mode is the official greedy vertex-cover approximation and the extent threshold is 0.10.


In [5]:
dime_minimal_df, dime_metrics_df, dime_top_df = run_dime_discovery(
    dime_df,
    RESULTS_DIR,
)

print('Minimal RFDs discovered by DiMε:', len(dime_minimal_df))
display(dime_top_df.loc[:, [
    'rule_label', 'g3_error', 'support', 'confidence',
    'baseline_confidence', 'lift'
]].round(4))


Minimal RFDs discovered by DiMε: 5


,rule_label,g3_error,support,confidence,baseline_confidence,lift
0,"station, PM10, NO2, O3, TEMP -> WSPM",0.0919,0.0016,0.8988,0.7730,1.1628
1,"PM2.5, PM10, NO2, O3, TEMP -> WSPM",0.0740,0.0013,0.8886,0.7691,1.1553
2,"station, PM2.5, NO2, O3, TEMP -> WSPM",0.0865,0.0014,0.8872,0.7705,1.1515
3,"station, PM2.5, PM10, O3, TEMP -> WSPM",0.0811,0.0013,0.8870,0.7734,1.1469
4,time_slot -> WSPM,0.0465,0.2496,0.8579,0.7707,1.1132


In [6]:
dime_rules = rules_from_discovery_frame(dime_top_df.head(4))
print('Rules selected for post-discovery robustness analysis:')
display(dime_top_df.head(4)[['rule_label', 'confidence', 'support', 'lift']].round(4))


Rules selected for post-discovery robustness analysis:


,rule_label,confidence,support,lift
0,"station, PM10, NO2, O3, TEMP -> WSPM",0.8988,0.0016,1.1628
1,"PM2.5, PM10, NO2, O3, TEMP -> WSPM",0.8886,0.0013,1.1553
2,"station, PM2.5, NO2, O3, TEMP -> WSPM",0.8872,0.0014,1.1515
3,"station, PM2.5, PM10, O3, TEMP -> WSPM",0.8870,0.0013,1.1469


## Step 5: Post-discovery analysis of DiMε RFDs

In [7]:
bootstrap_df = run_bootstrap_validation(
    dime_df,
    RESULTS_DIR / 'rfd_bootstrap_summary.csv',
    RESULTS_DIR / 'rfd_bootstrap_iterations.csv',
    rules=dime_rules,
)
train_test_df = run_train_test_validation(
    dime_df,
    RESULTS_DIR / 'rfd_train_test_validation.csv',
    rules=dime_rules,
)


## Step 6: Continuous profiling and violation analysis

In [8]:
window_df = run_window_evolution(
    dime_df,
    dime_metrics_df,
    RESULTS_DIR / 'rfd_window_evolution.csv',
    FIGURES_DIR / 'rfd_confidence_over_time.png',
    rules=dime_rules,
)
violations_summary_df = run_violation_analysis(
    dime_df,
    dime_metrics_df,
    RESULTS_DIR / 'rfd_violations_summary.csv',
    RESULTS_DIR / 'dime_top_violating_pairs.csv',
    FIGURES_DIR / 'rfd_violations_by_month_station.png',
    rules=dime_rules,
)
(RESULTS_DIR / 'rfd_top_violating_pairs.csv').write_bytes(
    (RESULTS_DIR / 'dime_top_violating_pairs.csv').read_bytes()
)


93029

In [9]:
print('Bootstrap summary')
display(bootstrap_df.round(4))
print('Temporal train-test validation')
display(train_test_df.round(4))
print('Window evolution')
display(window_df.loc[:, [
    'rule_label', 'window_start', 'antecedent_pairs', 'support', 'confidence',
    'violation_rate', 'baseline_confidence', 'baseline_confidence_std',
    'lift', 'abrupt_change'
]].round(4))
print('Violation concentration')
display(violations_summary_df.head(20).round(4))


Bootstrap summary


,rule_label,lhs,rhs,support_mean,support_std,support_ci95_low,support_ci95_high,confidence_mean,confidence_std,confidence_ci95_low,confidence_ci95_high,lift_mean,lift_std,lift_ci95_low,lift_ci95_high,iterations
0,"station, PM10, NO2, O3, TEMP -> WSPM","station, PM10, NO2, O3, TEMP",WSPM,0.0022,0.0001,0.0020,0.0024,0.9301,0.0101,0.9165,0.9538,1.2068,0.0187,1.1706,1.2436,30
1,"PM2.5, PM10, NO2, O3, TEMP -> WSPM","PM2.5, PM10, NO2, O3, TEMP",WSPM,0.0019,0.0001,0.0017,0.0020,0.9273,0.0109,0.9082,0.9465,1.2026,0.0220,1.1521,1.2338,30
2,"station, PM2.5, NO2, O3, TEMP -> WSPM","station, PM2.5, NO2, O3, TEMP",WSPM,0.0019,0.0001,0.0018,0.0021,0.9247,0.0114,0.9047,0.9485,1.1996,0.0230,1.1469,1.2433,30
3,"station, PM2.5, PM10, O3, TEMP -> WSPM","station, PM2.5, PM10, O3, TEMP",WSPM,0.0019,0.0001,0.0017,0.0020,0.9241,0.0098,0.9079,0.9455,1.1980,0.0198,1.1606,1.2297,30


Temporal train-test validation


,rule_label,lhs,rhs,support_train,confidence_train,violation_rate_train,baseline_confidence_train,baseline_confidence_std_train,lift_train,support_test,confidence_test,violation_rate_test,baseline_confidence_test,baseline_confidence_std_test,lift_test,confidence_delta_test_minus_train,lift_delta_test_minus_train
0,"station, PM10, NO2, O3, TEMP -> WSPM","station, PM10, NO2, O3, TEMP",WSPM,0.0016,0.8882,0.1118,0.7653,0.0159,1.1606,0.0021,0.9365,0.0635,0.7951,0.0320,1.1779,0.0483,0.0173
1,"PM2.5, PM10, NO2, O3, TEMP -> WSPM","PM2.5, PM10, NO2, O3, TEMP",WSPM,0.0012,0.8732,0.1268,0.7638,0.0191,1.1433,0.0019,0.9483,0.0517,0.8019,0.0296,1.1825,0.0751,0.0392
2,"station, PM2.5, NO2, O3, TEMP -> WSPM","station, PM2.5, NO2, O3, TEMP",WSPM,0.0014,0.8664,0.1336,0.7594,0.0139,1.1409,0.0016,0.9517,0.0483,0.7936,0.0430,1.1993,0.0853,0.0584
3,"station, PM2.5, PM10, O3, TEMP -> WSPM","station, PM2.5, PM10, O3, TEMP",WSPM,0.0012,0.8731,0.1269,0.7682,0.0165,1.1366,0.0017,0.9527,0.0473,0.8072,0.0276,1.1802,0.0796,0.0437


Window evolution


/var/folders/hn/yqq1wz1x1qg3gbdk_f7rxb5c0000gn/T/ipykernel_53548/2638387130.py:10: UserWarning: obj.round has no effect with datetime, timedelta, or period dtypes. Use obj.dt.round(...) instead.
  ]].round(4))


,rule_label,window_start,antecedent_pairs,support,confidence,violation_rate,baseline_confidence,baseline_confidence_std,lift,abrupt_change
0,"PM2.5, PM10, NO2, O3, TEMP -> WSPM",2013-02-01,1,0.0357,1.00,0.00,0.7000,0.4583,1.4286,False
1,"PM2.5, PM10, NO2, O3, TEMP -> WSPM",2013-03-01,1,0.0020,1.00,0.00,0.8000,0.4000,1.2500,False
2,"PM2.5, PM10, NO2, O3, TEMP -> WSPM",2013-04-01,4,0.0051,0.25,0.75,0.5833,0.2173,0.4286,True
3,"PM2.5, PM10, NO2, O3, TEMP -> WSPM",2013-05-01,2,0.0040,1.00,0.00,0.7167,0.3337,1.3953,True
4,"PM2.5, PM10, NO2, O3, TEMP -> WSPM",2013-06-01,1,0.0020,1.00,0.00,0.9667,0.1795,1.0345,False
...,...,...,...,...,...,...,...,...,...,...
191,"station, PM2.5, PM10, O3, TEMP -> WSPM",2016-10-01,10,0.0202,1.00,0.00,0.8800,0.1194,1.1364,False
192,"station, PM2.5, PM10, O3, TEMP -> WSPM",2016-11-01,8,0.0103,1.00,0.00,0.9292,0.0698,1.0762,False
193,"station, PM2.5, PM10, O3, TEMP -> WSPM",2016-12-01,9,0.0181,1.00,0.00,0.9889,0.0333,1.0112,False
194,"station, PM2.5, PM10, O3, TEMP -> WSPM",2017-01-01,6,0.0077,1.00,0.00,0.9222,0.1117,1.0843,False


Violation concentration


,rule_label,station,month,time_slot,violation_pairs,mean_rhs_abs_diff,max_rhs_abs_diff,share_within_rule
0,"PM2.5, PM10, NO2, O3, TEMP -> WSPM",Changping,2016-01,afternoon,11,2.0364,2.90,0.0556
1,"PM2.5, PM10, NO2, O3, TEMP -> WSPM",Changping,2013-12,morning,7,2.0286,2.35,0.0354
2,"PM2.5, PM10, NO2, O3, TEMP -> WSPM",Changping,2014-12,afternoon,7,1.8571,2.30,0.0354
3,"PM2.5, PM10, NO2, O3, TEMP -> WSPM",Changping,2013-12,afternoon,7,1.5714,2.35,0.0354
4,"PM2.5, PM10, NO2, O3, TEMP -> WSPM",Changping,2014-12,morning,6,1.4417,1.80,0.0303
5,"PM2.5, PM10, NO2, O3, TEMP -> WSPM",Aotizhongxin,2016-02,night,6,1.3333,1.45,0.0303
6,"PM2.5, PM10, NO2, O3, TEMP -> WSPM",Changping,2016-02,morning,6,1.2583,1.40,0.0303
7,"PM2.5, PM10, NO2, O3, TEMP -> WSPM",Changping,2013-11,afternoon,6,1.2167,1.50,0.0303
8,"PM2.5, PM10, NO2, O3, TEMP -> WSPM",Aotizhongxin,2013-12,afternoon,5,1.8200,2.05,0.0253
9,"PM2.5, PM10, NO2, O3, TEMP -> WSPM",Aotizhongxin,2016-02,morning,5,1.0800,1.20,0.0253
